# actas-cnn — harness de optimizacion de HP (Kaggle / Colab)

Notebook dedicado al loop de busqueda inspirado en `karpathy/autoresearch`.
El agente Claude propone una `config_XXX.json`; este notebook la ejecuta en
GPU y agrega el resultado al `results.tsv` del repo. El humano descarga
el TSV actualizado y lo committea de regreso al branch.

**Como correr:**
1. `Runtime` -> `Change runtime type` -> **GPU T4** (Colab) o habilitar GPU en Kaggle.
2. Ajustar `CONFIG_FILE` (celda 5) al archivo `configs/run_XXX.json` que toca.
3. `Runtime` -> `Run all`. Tiempo esperado: ~4-6 min en T4 (20 epochs ResNet-18 + eval).

**Output:** una nueva fila en `results.tsv`. El notebook lo muestra al final y lo descarga al navegador.

## 1. Clonar el repo (branch del harness)

In [ ]:
import os, sys

REPO_URL = 'https://github.com/f3r21/actas-cnn.git'
REPO_DIR = 'actas-cnn'
BRANCH = 'main'  # cambiar a 'harness/autoresearch-loop' si el PR aun no esta merged

if not os.path.isdir(REPO_DIR):
    os.system(f'git clone --branch {BRANCH} {REPO_URL}')
os.chdir(REPO_DIR)
sys.path.insert(0, os.getcwd())
os.system('git log --oneline -3')
print('cwd:', os.getcwd())

## 2. Instalar dependencias

In [ ]:
os.system('pip install -q -r requirements.txt')

## 3. Descargar bundle de datos (~460 MB) desde HF

Publico, sin token. Incluye los crops y manifests de los 3 splits.

In [ ]:
from huggingface_hub import hf_hub_download
from config import REMOTE

bundle = hf_hub_download(
    repo_id=REMOTE.hf_dataset_repo,
    repo_type='dataset',
    filename='data_bundle.tar.gz',
)
print(f'bundle: {bundle}')
os.system(f'tar -xzf {bundle} -C .')
os.system('ls data/manifest_*.csv && ls data/crops_train/ | head -3')

## 4. Verificar GPU

In [ ]:
import torch
from env import torch_device
device = torch_device()
print(f'device: {device}')
if device.type == 'cuda':
    print(f'gpu: {torch.cuda.get_device_name(0)}')

## 5. Elegir la config a correr

Apuntar a un JSON en `configs/`. Si quieres correr varias en serie, pasa
una lista en `CONFIG_FILES`.

In [ ]:
# Una sola config
CONFIG_FILES = ['configs/baseline_full.json']

# Para corridas en serie overnight, descomentar y editar:
# CONFIG_FILES = [
#     'configs/run_001_ls01.json',
#     'configs/run_002_randaug.json',
#     'configs/run_003_mixup02.json',
# ]

for cfg in CONFIG_FILES:
    assert os.path.isfile(cfg), f'falta {cfg}'
print(f'configs a ejecutar: {len(CONFIG_FILES)}')
for cfg in CONFIG_FILES:
    print(f'  {cfg}')

## 6. Ejecutar los runs

Cada run: train (20 epochs) + evaluate (val) + append a `results.tsv`.
~4-6 min/run en T4. El runner imprime al final una linea con el composite
y la decision keep/discard.

In [ ]:
for cfg in CONFIG_FILES:
    print(f'\n{"="*70}\n[notebook] corriendo {cfg}\n{"="*70}')
    rc = os.system(f'python scripts/run_experiment.py --config {cfg}')
    if rc != 0:
        print(f'[notebook] WARN: run_experiment.py exit {rc} para {cfg}')
print('\n[notebook] todos los runs terminaron')

## 7. Mostrar `results.tsv` (ultimas filas)

In [ ]:
import pandas as pd
df = pd.read_csv('results.tsv', sep='\t')
print(f'total runs registrados: {len(df)}')
print('\nTop 5 por composite:')
print(df.sort_values('composite', ascending=False).head(5).to_string(index=False))
print('\nUltimas 5 filas:')
print(df.tail(5).to_string(index=False))

## 8. Descargar `results.tsv` al navegador

Pegar el TSV actualizado de regreso al repo local y hacer commit en la
branch del harness.

In [ ]:
try:
    from google.colab import files
    files.download('results.tsv')
    print('descargado al navegador')
except ImportError:
    print('no estamos en Colab; archivo disponible en results.tsv')
    print('en Kaggle: usar el panel Output -> Download para bajarlo')

## 9. (Opcional) Descargar los checkpoints generados

Util si quieres conservar el best del run para evaluarlo despues con
`evaluate_with_solver.py` o sobre test.

In [ ]:
import glob
ckpts = sorted(glob.glob('checkpoints/resnet18_*_best.pt'))
ckpts = [c for c in ckpts if 'resnet18_best' not in c]  # excluir baseline
print(f'checkpoints generados ({len(ckpts)}):')
for c in ckpts:
    print(f'  {c}')

try:
    from google.colab import files
    for c in ckpts:
        files.download(c)
except ImportError:
    print('en Kaggle: panel Output para bajar los .pt')